# Poglavje 7: Gradnja klepetalnih aplikacij
## Microsoft Foundry Models API hitri začetek

Ta zvezek je prilagojen iz [Azure OpenAI vzorčnih repozitorijev](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ki vključuje zvezke, ki dostopajo do storitev [Azure OpenAI](notebook-azure-openai.ipynb).

> **Opomba:** GitHub Models se upokojuje konec julija 2026. Ta zvezek zdaj cilja na [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), ki ponuja enak katalog modelov za brezplačno preizkušanje in izkušnjo Azure AI Inference SDK.


# Pregled  
"Veliki jezikovni modeli so funkcije, ki preslikajo besedilo v besedilo. Glede na vhodni niz besedila velik jezikovni model poskuša napovedati besedilo, ki bo sledilo"(1). Ta zvezek »hitri začetek« bo uporabnikom predstavil osnovne koncepte LLM, ključne zahteve paketov za začetek z AML, rahlo uvod v oblikovanje pozivov ter nekaj kratkih primerov različnih primerov uporabe. 


## Kazalo vsebine  

[Pregled](#overview)  
[Kako uporabljati OpenAI storitev](#how-to-use-openai-service)  
[1. Ustvarjanje vaše OpenAI storitve](#1.-creating-your-openai-service)  
[2. Namestitev](#2.-installation)    
[3. Pooblastila](#3.-credentials)  

[Uporabniški primeri](#use-cases)    
[1. Povzetek besedila](#1.-summarize-text)  
[2. Razvrščanje besedila](#2.-classify-text)  
[3. Generiranje novih imen produktov](#3.-generate-new-product-names)  
[4. Fino prilagajanje klasifikatorja](#4.fine-tune-a-classifier)  

[Reference](#references)


### Ustvarite svoj prvi poziv  
Ta kratka vaja bo zagotovila osnovno uvod v pošiljanje pozivov modelu v Microsoft Foundry Models za enostavno nalogo "povzemanje".


**Koraki**:  
1. Namestite knjižnico `azure-ai-inference` v vaše Python okolje, če tega še niste storili.  
2. Naložite standardne pomočne knjižnice in nastavite svoje poverilnice za Microsoft Foundry Models.  
3. Izberite model za vašo nalogo  
4. Ustvarite preprost poziv za model  
5. Pošljite vaš zahtevek API-ju modela!


### 1. Namestite `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Uvozite pomožne knjižnice in ustvarite poverilnice


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Iskanje pravega modela  
Modeli, kot sta GPT-4o in GPT-4o mini, lahko razumejo in generirajo naravni jezik ter so na voljo v katalogu Microsoft Foundry Models skupaj z modeli podjetij Meta, Mistral, Cohere in Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Oblikovanje pozivov  

"Čarobnost velikih jezikovnih modelov je, da se z učenjem minimiziranja te napake pri napovedovanju na ogromnih količinah besedila modeli naučijo konceptov, ki so koristni za te napovedi. Na primer, naučijo se konceptov, kot so"(1):

* kako se črkuje
* kako deluje slovnica
* kako parafrazirati
* kako odgovarjati na vprašanja
* kako voditi pogovor
* kako pisati v mnogih jezikih
* kako programirati
* itd.

#### Kako nadzorovati velik jezikovni model  
"Od vseh vhodov velikemu jezikovnemu modelu je daleč najbolj vpliven besedilni poziv(1).

Veliki jezikovni modeli lahko s pozivi usmerjamo na več načinov:

Navodilo: Povej modelu, kaj želiš
Dokončanje: Napovej modelu, naj dokonča začetek tega, kar želiš
Demonstracija: Pokaži modelu, kaj želiš, z:
Nekaj primeri v pozivu
Več sto ali tisoč primeri v podatkovnem naboru za fino urjenje"



#### Obstajajo trije osnovni nasveti za ustvarjanje pozivov:

**Pokaži in povej**. Jasno sporoči, kaj želiš, bodisi z navodili, primeri ali kombinacijo obojega. Če želiš, da model uredi seznam elementov po abecedi ali razvrsti odstavek po čustvenem tonu, mu pokaži, da je to to, kar želiš.

**Nudi kakovostne podatke**. Če želiš zgraditi razvrščevalnik ali doseči, da model sledi vzorcu, poskrbi za dovolj primerov. Preveri svoje primere — model je ponavadi dovolj pameten, da prepozna osnovne pravopisne napake in ti da odgovor, lahko pa tudi predpostavi, da je to namerno, kar lahko vpliva na odgovor.

**Preveri nastavitve.** Temperatura in top_p nastavitve nadzorujejo, kako determinističen je model pri ustvarjanju odgovora. Če pričakuješ odgovor, kjer je samo en pravilen odgovor, želiš nastavitve nastaviti nižje. Če iščeš bolj raznolike odgovore, jih lahko nastaviš višje. Najpogostejša napaka pri teh nastavitvah je domneva, da nadzorujejo "bistroumnost" ali "kreativnost".


Vir: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Oddaj!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ponovite isti klic, kako se rezultati primerjajo?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Povzetek besedila  
#### Izziv  
Povzemite besedilo tako, da na konec besedilnega odlomka dodate 'tl;dr:'. Opazite, kako model razume, kako izvesti številne naloge brez dodatnih navodil. Lahko eksperimentirate z bolj opisnimi pozivi kot tl;dr, da spremenite vedenje modela in prilagodite povzemanje, ki ga prejmete(3).  

Nedavno delo je pokazalo pomembne izboljšave pri mnogih nalogah in referenčnih testih obdelave naravnega jezika z vnaprejšnjim učenjem na velikem korpusu besedila, slednjo pa fino prilagajanje na določeno nalogo. Čeprav je arhitektura običajno nalogi nepristranska, ta metoda še vedno zahteva specifične podatkovne zbirke za fino prilagajanje naloge s tisoči ali deset tisoči primerov. Nasprotno, ljudje lahko na splošno izvedejo novo jezikovno nalogo že iz le nekaj primerov ali preprostih navodil – nekaj, s čimer sodobni sistemi obdelave naravnega jezika še vedno precej težko upravljajo. Tukaj pokažemo, da povečanje velikosti jezikovnih modelov močno izboljša nepristransko izvajanje nalog z malo primeri, včasih celo doseže konkurenco z dosedanjimi najboljšimi pristopi fino prilagajanja.  



Povzetek  


# Vaje za več uporabe  
1. Povzemanje besedila  
2. Razvrščanje besedila  
3. Generiranje novih imen izdelkov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasificiraj besedilo  
#### Izziv  
Razvrsti predmete v kategorije, podane ob času sklepanja. V naslednjem primeru zagotovimo tako kategorije kot besedilo za klasifikacijo v pozivu (*playground_reference). 

Povpraševanje stranke: Pozdravljeni, ena tipka na tipkovnici mojega prenosnika se je pred kratkim zlomila in potreboval bom zamenjavo:

Klasificirana kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Ustvarite nove imena izdelkov
#### Izziv
Ustvarite imena izdelkov iz primerov besed. Tukaj v poziv vključimo informacije o izdelku, za katerega bomo ustvarjali imena. Prav tako ponudimo podoben primer, da pokažemo vzorec, ki ga želimo dobiti. Nastavili smo tudi visoko vrednost temperature, da povečamo naključnost in povemo bolj inovativne odgovore.

Opis izdelka: domači pripravljalnik mlečnih napitkov
Semenske besede: hiter, zdrav, kompakten.
Imenu izdelka: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis izdelka: par čevljev, ki lahko ustrezajo katerikoli velikosti stopala.
Semenske besede: prilagodljiv, prilegajoč, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najboljše prakse za fino uglaševanje GPT modelov za razvrščanje besedila](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za več pomoči  
[Ekipa za komercializacijo OpenAI](AzureOpenAITeam@microsoft.com) 


# Sodelujoči
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
